# Imputation Forward

Ce notebook :
1. Charge la base `base_avec_statut.parquet` depuis MinIO (couche Gold)
2. Identifie les agents hors barème et compte les mois incohérents
3. Applique une **imputation forward** (dernière valeur valide connue) sous conditions
4. Produit des statistiques descriptives et un export Excel


In [ ]:
import io
import os
import polars as pl
import urllib3
from dotenv import load_dotenv
import pandas as pd
import boto3
from botocore.client import Config

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv(".env")

s3 = boto3.client(
    "s3",
    endpoint_url          = os.getenv("MINIO_ENDPOINT", "http://192.168.1.230:30137"),
    aws_access_key_id     = os.getenv("MINIO_ACCESS_KEY", "datalab-team"),
    aws_secret_access_key = os.getenv("MINIO_SECRET_KEY", "minio-datalabteam123"),
    config                = Config(signature_version="s3v4"),
    region_name           = "us-east-1",
    verify                = False,
)

buf = io.BytesIO()
s3.download_fileobj("gold", "panel_admin/base_avec_statut.parquet", buf)
buf.seek(0)
base = pl.read_parquet(buf)

print(f"Base chargée : {base.shape[0]:,} lignes × {base.shape[1]} colonnes")
print(f"Agents uniques : {base.select('cle_unique').n_unique():,}")
base.head(3)


Base chargée : 6,914,095 lignes × 55 colonnes
Agents uniques : 317,037


cle_unique,matricule,nom,sexe,date_naissance,situation_matrimoniale,nombre_enfant,situation_normalisee,CODE_ORGANISME,organisme,CODE_EMPLOI,emploi,Code_CITP,Metier_CITP,CODE_GRADE,grade,statut_fonctionnaire,CODE_FONCTION,fonction,CODE_SERVICE,service,CODE_AFFECTATION,lieu_affectation,CODE_POSTE,prise_service,date_retraite,age_retraite,GRADE,GRADE_SOURCE,mois_annee,annee,mois,montant_brut,montant_net,retenue_pension,impot,charge_patronale,montant_brut_SOURCE,montant_net_SOURCE,fichier_source,NB_POSTES,montant_brut_total,montant_brut_bareme,montant_net_total,ANNEE,MOIS_NUM,MOIS,SALAIRE_BRUT,SEXE_STD,bareme,grade_ref,bareme_detail,min_bareme,max_bareme,Statut
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,f64,f64,f64,f64,f64,str,str,str,u32,f64,f64,f64,i32,i32,str,f64,str,str,str,str,f64,f64,str
"""100604R18""","""100604R""","""EDI ASSIRO MATHIAS""","""Masculin""","""1959-07-26""","""Célibataire""","""0""","""en_activite""","""18""","""Min. d'Etat de l'Agriculture, …","""03DF""","""Ingénieur Agronome""","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A4""","""063""","""Représentation Groupe 3""","""1250""","""Direction Régionale""","""2AA""","""Abengourou""",null,"""2003-01-02""",null,null,"""A4""","""OBSERVE""","""012024""",2024,1,732455.0,590594.0,40741.0,94116.0,81530.908764,"""OBSERVE""","""OBSERVE""","""012024.xlsx""",1,732455.0,632455.0,590594.0,2024,1,"""Janvier""",632455.0,"""Homme""","""Barême Général""","""A4""","""Barême Général""",340285.0,662454.0,"""Conforme au barème"""
"""100604R18""","""100604R""","""EDI ASSIRO MATHIAS""","""Masculin""","""1959-07-26""","""Célibataire""","""0""","""en_activite""","""18""","""Min. d'Etat de l'Agriculture, …","""03DF""","""Ingénieur Agronome""","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A4""","""063""","""Représentation Groupe 3""","""1250""","""Direction Régionale""","""2AA""","""Abengourou""",null,"""2003-01-02""",null,null,"""A4""","""OBSERVE""","""022024""",2024,2,732455.0,590594.0,40741.0,94116.0,81530.908764,"""OBSERVE""","""OBSERVE""","""022024.xlsx""",1,732455.0,632455.0,590594.0,2024,2,"""Février""",632455.0,"""Homme""","""Barême Général""","""A4""","""Barême Général""",340285.0,662454.0,"""Conforme au barème"""
"""100604R18""","""100604R""","""EDI ASSIRO MATHIAS""","""Masculin""","""1959-07-26""","""Célibataire""","""0""","""en_activite""","""18""","""Min. d'Etat de l'Agriculture, …","""03DF""","""Ingénieur Agronome""","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A4""","""063""","""Représentation Groupe 3""","""1250""","""Direction Régionale""","""2AA""","""Abengourou""",null,"""2003-01-02""",null,null,"""A4""","""OBSERVE""","""032024""",2024,3,732455.0,590594.0,40741.0,94116.0,81530.908764,"""OBSERVE""","""OBSERVE""","""032024.xlsx""",1,732455.0,632455.0,590594.0,2024,3,"""Mars""",632455.0,"""Homme""","""Barême Général""","""A4""","""Barême Général""",340285.0,662454.0,"""Conforme au barème"""


## 1. Exploration des agents hors barème


In [2]:
statuts_incoherents = [
    "Inférieur au minimum du barème",
    "Supérieur au maximum du barème"
]

hors = base.filter(pl.col("Statut").is_in(statuts_incoherents))
print(f"Lignes hors barème : {len(hors):,} / {len(base):,} ({len(hors)/len(base)*100:.2f}%)")
print(f"Agents concernés  : {hors.select('cle_unique').n_unique():,}")
hors.head(5)


Lignes hors barème : 447,602 / 6,914,095 (6.47%)
Agents concernés  : 117,070


cle_unique,matricule,nom,sexe,date_naissance,situation_matrimoniale,nombre_enfant,situation_normalisee,CODE_ORGANISME,organisme,CODE_EMPLOI,emploi,Code_CITP,Metier_CITP,CODE_GRADE,grade,statut_fonctionnaire,CODE_FONCTION,fonction,CODE_SERVICE,service,CODE_AFFECTATION,lieu_affectation,CODE_POSTE,prise_service,date_retraite,age_retraite,GRADE,GRADE_SOURCE,mois_annee,annee,mois,montant_brut,montant_net,retenue_pension,impot,charge_patronale,montant_brut_SOURCE,montant_net_SOURCE,fichier_source,NB_POSTES,montant_brut_total,montant_brut_bareme,montant_net_total,ANNEE,MOIS_NUM,MOIS,SALAIRE_BRUT,SEXE_STD,bareme,grade_ref,bareme_detail,min_bareme,max_bareme,Statut
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,f64,f64,f64,f64,f64,str,str,str,u32,f64,f64,f64,i32,i32,str,f64,str,str,str,str,f64,f64,str
"""108076E31""","""108076E""","""HEMA CATHERINE""","""Féminin""","""1959-05-26""","""Célibataire""","""0""","""en_activite""","""31""","""Min. des Eaux et Forêts""","""9CEU""","""INGENIEUR PRINCIPAL: EAUX ET F…","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A5""",null,""" ""","""0311""","""D A A F""","""1AA""","""Abidjan""",null,"""1977-10-01""",null,null,"""A5""","""OBSERVE""","""022024""",2024,2,1.135876e6,786287.0,41312.0,95800.0,82673.594238,"""OBSERVE""","""OBSERVE""","""022024.xlsx""",1,1.135876e6,935876.0,786287.0,2024,2,"""Février""",935876.0,"""Femme""","""Barême Général""","""A5""","""Barême Général""",379078.0,687823.0,"""Supérieur au maximum du barème"""
"""110082J22""","""110082J""","""KOUAO BROU""","""Masculin""","""1958-12-31""","""Marié""","""0""","""en_activite""","""22""","""Min. de l'Education Nationale …","""25WU""","""Inspecteur d'Enseignement Prim…","""2351""","""Spécialistes des méthodes péda…","""12""","""Classe Exceptionnelle 2ème Ech…","""Fonctionnaire Catégorie A4""","""054""","""Inspecteur Primaire""","""22BU""","""IEP Bongouanou""","""3CA""","""Bongouanou""",null,"""1978-01-07""",null,null,"""A4""","""OBSERVE""","""082024""",2024,8,1.021375e6,770918.0,59976.0,138890.0,120024.0,"""OBSERVE""","""OBSERVE""","""082024.xlsx""",1,1.021375e6,999375.0,770918.0,2024,8,"""Août""",999375.0,"""Homme""","""ENS SEC & PRIM""","""A4""","""ENS SEC & PRIM_SEC""",457071.0,819511.0,"""Supérieur au maximum du barème"""
"""112035G12""","""112035G""","""YAO YAO ADOLPHE""","""Masculin""","""1961-02-11""","""Marié""","""0""","""en_activite""","""12""","""Ministere des Finances et du B…","""9AAB""","""ADMINISTRATEUR GENERAL SERV.FI…","""1112 | 1211""","""Hauts fonctionnaires | Directe…","""31""","""1ère Classe 1er Echelon""","""Fonctionnaire Catégorie A7""","""749""","""Régisseur 1 Milliard - gr 2""","""1216""","""D.G.T.C.P""","""4SA""","""Seguela""",null,"""1995-01-03""",null,null,"""A7""","""OBSERVE""","""012024""",2024,1,957592.0,821044.0,42200.0,87344.0,84450.660264,"""OBSERVE""","""OBSERVE""","""012024.xlsx""",1,957592.0,582592.0,821044.0,2024,1,"""Janvier""",582592.0,"""Homme""","""Barême Général""","""A7""","""Barême Général""",646210.0,750915.0,"""Inférieur au minimum du barème"""
"""112035G12""","""112035G""","""YAO YAO ADOLPHE""","""Masculin""","""1961-02-11""","""Marié""","""0""","""en_activite""","""12""","""Ministere des Finances et du B…","""9AAB""","""ADMINISTRATEUR GENERAL SERV.FI…","""1112 | 1211""","""Hauts fonctionnaires | Directe…","""32""","""1ère Classe 2ème Echelon""","""Fonctionnaire Catégorie A7""","""749""","""Régisseur 1 Milliard - gr 2""","""1216""","""D.G.T.C.P""","""4SA""","""Seguela""",null,"""1995-01-03""",null,null,"""A7""","""OBSERVE""","""032024""",2024,3,980411.0,837443.0,43853.0,92111.0,87758.644658,"""OBSERVE""","""OBSERVE""","""032024.xlsx""",1,980411.0,605411.0,837443.0,2024,3,"""Mars""",605411.0,"""Homme""","""Barême Général""","""A7""","""Barême Général""",646210.0,750915.0,"""Inférieur au minimum du barème"""
"""112035G12""","""112035G""","""

In [3]:
# Nombre de mois incohérents par agent (optimisé : .is_in().sum() au lieu de when/then)
mois_incoherents = (
    base.group_by("cle_unique")
    .agg(
        pl.col("Statut").is_in(statuts_incoherents).sum().alias("nombre_de_mois_incoherents")
    )
    .filter(pl.col("nombre_de_mois_incoherents") > 0)
)

print(f"Agents avec au moins 1 mois incohérent : {len(mois_incoherents):,}")
mois_incoherents.head(5)


Agents avec au moins 1 mois incohérent : 117,070


cle_unique,nombre_de_mois_incoherents
str,u32
"""869624A24""",17
"""438696T12""",1
"""238003F22""",1
"""919222Z23""",2
"""289481X22""",1


In [4]:
# Répartition : combien d'agents par nombre de mois incohérents
table_repartition = (
    mois_incoherents
    .group_by("nombre_de_mois_incoherents")
    .agg(pl.col("cle_unique").n_unique().alias("nb_agents"))
    .sort("nombre_de_mois_incoherents")
    .with_columns(
        (pl.col("nb_agents") * pl.col("nombre_de_mois_incoherents")).alias("nb_lignes_incoherentes")
    )
)
print(table_repartition)

# Total des lignes potentiellement imputables (1 à 5 mois)
total_imputable = table_repartition.filter(pl.col("nombre_de_mois_incoherents") <= 5)
print(f"\nTotal lignes imputables (1-5 mois) : {total_imputable.select(pl.col('nb_lignes_incoherentes').sum()).item():,}")
print(f"Total agents imputables (1-5 mois) : {total_imputable.select(pl.col('nb_agents').sum()).item():,}")


shape: (24, 3)
┌────────────────────────────┬───────────┬────────────────────────┐
│ nombre_de_mois_incoherents ┆ nb_agents ┆ nb_lignes_incoherentes │
│ ---                        ┆ ---       ┆ ---                    │
│ u32                        ┆ u32       ┆ u32                    │
╞════════════════════════════╪═══════════╪════════════════════════╡
│ 1                          ┆ 66033     ┆ 66033                  │
│ 2                          ┆ 15029     ┆ 30058                  │
│ 3                          ┆ 5129      ┆ 15387                  │
│ 4                          ┆ 3317      ┆ 13268                  │
│ 5                          ┆ 3420      ┆ 17100                  │
│ …                          ┆ …         ┆ …                      │
│ 20                         ┆ 154       ┆ 3080                   │
│ 21                         ┆ 195       ┆ 4095                   │
│ 22                         ┆ 386       ┆ 8492                   │
│ 23                         ┆ 48

## 2. Méthodologie d'imputation — Forward Fill

**Principe** : on remplace le montant d'un mois hors barème par la dernière valeur valide connue pour le même agent.

**Conditions d'éligibilité** (toutes doivent être remplies) :
- Le nombre de mois hors barème est entre **1 et 5**
- Le **grade** (GRADE) reste **inchangé** sur toute la période
- L'**emploi** reste **inchangé** sur toute la période
- L'**organisme** (CODE_ORGANISME) reste **inchangé** sur toute la période

Si le premier mois est hors barème (pas de valeur précédente), on utilise un **backward fill** en complément.


In [5]:
# ─────────────────────────────────────────────────────
# ÉTAPE 1 — Calcul des critères d'éligibilité par agent
# ─────────────────────────────────────────────────────
criteria = (
    base.group_by("cle_unique")
    .agg([
        pl.col("Statut").is_in(statuts_incoherents).sum().alias("nb_incoh"),
        pl.col("GRADE").n_unique().alias("n_grade"),
        pl.col("emploi").n_unique().alias("n_emploi"),
        pl.col("CODE_ORGANISME").n_unique().alias("n_org"),
    ])
    .with_columns(
        (
            (pl.col("nb_incoh") > 0) &
            (pl.col("nb_incoh") <= 5) &
            (pl.col("n_grade") == 1) &
            (pl.col("n_emploi") == 1) &
            (pl.col("n_org") == 1)
        ).alias("eligible_imputation")
    )
)

nb_eligible = criteria.filter(pl.col("eligible_imputation")).shape[0]
nb_total_incoh = criteria.filter(pl.col("nb_incoh") > 0).shape[0]
print(f"Agents hors barème  : {nb_total_incoh:,}")
print(f"Agents éligibles    : {nb_eligible:,}")
print(f"Agents non éligibles: {nb_total_incoh - nb_eligible:,}")


Agents hors barème  : 117,070
Agents éligibles    : 74,600
Agents non éligibles: 42,470


In [6]:
# ─────────────────────────────────────────────────────
# ÉTAPE 2 — Join + Forward Fill (avec .over() pour scoper par agent)
# ─────────────────────────────────────────────────────
df = base.join(criteria, on="cle_unique", how="left")

base_corrigee = (
    df
    .sort(["cle_unique", "mois_annee"])
    # Créer une colonne temporaire : Null pour les mois à imputer, valeur valide sinon
    .with_columns(
        pl.when(
            pl.col("eligible_imputation") &
            pl.col("Statut").is_in(statuts_incoherents)
        )
        .then(None)
        .otherwise(pl.col("montant_brut_bareme"))
        .alias("valeur_clean")
    )
    # Appliquer le forward fill SCOPED par agent (.over("cle_unique"))
    .with_columns(
        pl.when(pl.col("eligible_imputation"))
        .then(
            pl.col("valeur_clean")
            .fill_null(strategy="forward")
            .fill_null(strategy="backward")
            .over("cle_unique")
        )
        .otherwise(pl.col("montant_brut_bareme"))
        .alias("montant_corrige")
    )
)

print("✅ Forward fill terminé avec succès")
print(f"   Lignes totales : {len(base_corrigee):,}")


✅ Forward fill terminé avec succès
   Lignes totales : 6,914,095


## 3. Vérifications post-imputation


In [7]:
# Vérification : reste-t-il des Null dans montant_corrige ?
check_nulls = base_corrigee.select([
    pl.col("montant_corrige").is_null().sum().alias("nb_null"),
    (pl.col("montant_corrige").is_null().mean() * 100).alias("pct_null")
])
print("Vérification des Null restants :")
print(check_nulls)


Vérification des Null restants :
shape: (1, 2)
┌─────────┬──────────┐
│ nb_null ┆ pct_null │
│ ---     ┆ ---      │
│ u32     ┆ f64      │
╞═════════╪══════════╡
│ 11468   ┆ 0.165864 │
└─────────┴──────────┘


In [8]:
# Vérification par statut (conforme vs incohérent)
verif_statut = (
    base_corrigee
    .group_by(
        pl.col("Statut").is_in(statuts_incoherents).alias("est_incoherent")
    )
    .agg([
        pl.col("montant_corrige").is_null().sum().alias("nb_null"),
        pl.len().alias("nb_lignes")
    ])
)
print("Vérification par statut :")
print(verif_statut)


Vérification par statut :
shape: (2, 3)
┌────────────────┬─────────┬───────────┐
│ est_incoherent ┆ nb_null ┆ nb_lignes │
│ ---            ┆ ---     ┆ ---       │
│ bool           ┆ u32     ┆ u32       │
╞════════════════╪═════════╪═══════════╡
│ false          ┆ 0       ┆ 6466493   │
│ true           ┆ 11468   ┆ 447602    │
└────────────────┴─────────┴───────────┘


In [9]:
# Aperçu : lignes corrigées (éligibles + hors barème) avec avant/après
apercu_corrections = (
    base_corrigee
    .filter(
        pl.col("eligible_imputation") &
        pl.col("Statut").is_in(statuts_incoherents)
    )
    .select([
        "cle_unique", "mois_annee", "Statut", "nb_incoh",
        "montant_brut_bareme", "montant_corrige"
    ])
)
print(f"Nombre de lignes effectivement corrigées : {len(apercu_corrections):,}")
apercu_corrections.head(10)


Nombre de lignes effectivement corrigées : 115,504


cle_unique,mois_annee,Statut,nb_incoh,montant_brut_bareme,montant_corrige
str,str,str,u32,f64,f64
"""108076E31""","""022024""","""Supérieur au maximum du barème""",1,935876.0,695142.0
"""110082J22""","""082024""","""Supérieur au maximum du barème""",1,999375.0,773638.0
"""110439U22""","""052025""","""Supérieur au maximum du barème""",1,5.665764e6,null
"""110635Y43""","""012025""","""Supérieur au maximum du barème""",1,6.710194e6,1.19624e6
"""114115F22""","""072024""","""Supérieur au maximum du barème""",1,910233.0,788638.0
"""118057U25""","""062024""","""Inférieur au minimum du barème""",1,228200.0,null
"""120161X22""","""082024""","""Supérieur au maximum du barème""",1,1.319248e6,755448.0
"""122522F25""","""052024""","""Inférieur au minimum du barème""",1,226679.0,null
"""122551L25""","""052024""","""Inférieur au minimum du barème""",1,91280.0,null


## 4. Statistiques descriptives


In [10]:
# ═══════════════════════════════════════════════════════
# STATISTIQUES GLOBALES
# ═══════════════════════════════════════════════════════

stats_globales = base_corrigee.select([
    pl.len().alias("nb_lignes_total"),
    pl.col("cle_unique").n_unique().alias("nb_agents_total"),
    pl.col("Statut").is_in(statuts_incoherents).sum().alias("nb_lignes_incoherentes"),
    (pl.col("Statut").is_in(statuts_incoherents).mean() * 100).alias("pct_lignes_incoherentes"),
    pl.col("montant_corrige").is_null().sum().alias("nb_null_apres_imputation"),
])

print("📊 RÉSUMÉ GLOBAL")
print(stats_globales)

# ═══════════════════════════════════════════════════════
# IMPACT DE L'IMPUTATION
# ═══════════════════════════════════════════════════════

# Compter les lignes (éligibles ET hors barème) => c'est ce qui a été corrigé
nb_lignes_corrigees = base_corrigee.filter(
    pl.col("eligible_imputation") & pl.col("Statut").is_in(statuts_incoherents)
).shape[0]

# Compter les lignes (non éligibles ET hors barème) => non corrigées
nb_lignes_non_corrigees = base_corrigee.filter(
    ~pl.col("eligible_imputation") & pl.col("Statut").is_in(statuts_incoherents)
).shape[0]

print(f"\n📈 IMPACT IMPUTATION")
print(f"   Lignes hors barème corrigées     : {nb_lignes_corrigees:,}")
print(f"   Lignes hors barème NON corrigées : {nb_lignes_non_corrigees:,}")
print(f"   Total lignes hors barème         : {nb_lignes_corrigees + nb_lignes_non_corrigees:,}")

# ═══════════════════════════════════════════════════════
# COMPARAISON AVANT / APRÈS pour les lignes corrigées
# ═══════════════════════════════════════════════════════

comparaison = base_corrigee.filter(
    pl.col("eligible_imputation") & pl.col("Statut").is_in(statuts_incoherents)
).select([
    pl.col("montant_brut_bareme").mean().alias("moyenne_avant"),
    pl.col("montant_corrige").mean().alias("moyenne_apres"),
    pl.col("montant_brut_bareme").median().alias("mediane_avant"),
    pl.col("montant_corrige").median().alias("mediane_apres"),
])

print("\n📉 COMPARAISON AVANT / APRÈS (lignes corrigées uniquement)")
print(comparaison)

# ═══════════════════════════════════════════════════════
# RÉSUMÉ PAR AGENT (1 à 5 mois incohérents)
# ═══════════════════════════════════════════════════════

resume_agents = (
    base_corrigee
    .group_by("cle_unique")
    .agg([
        pl.col("nb_incoh").first(),
        pl.col("eligible_imputation").first(),
        pl.col("montant_corrige").is_null().sum().alias("null_restants")
    ])
    .filter(pl.col("eligible_imputation"))
)

print(f"\n👤 RÉSUMÉ AGENTS ÉLIGIBLES")
print(f"   Agents éligibles : {len(resume_agents):,}")
print(f"   Null restants    : {resume_agents.select(pl.col('null_restants').sum()).item()}")


📊 RÉSUMÉ GLOBAL
shape: (1, 5)
┌─────────────────┬─────────────────┬────────────────────┬────────────────────┬────────────────────┐
│ nb_lignes_total ┆ nb_agents_total ┆ nb_lignes_incohere ┆ pct_lignes_incoher ┆ nb_null_apres_impu │
│ ---             ┆ ---             ┆ ntes               ┆ entes              ┆ tation             │
│ u32             ┆ u32             ┆ ---                ┆ ---                ┆ ---                │
│                 ┆                 ┆ u32                ┆ f64                ┆ u32                │
╞═════════════════╪═════════════════╪════════════════════╪════════════════════╪════════════════════╡
│ 6914095         ┆ 317037          ┆ 447602             ┆ 6.473761           ┆ 11468              │
└─────────────────┴─────────────────┴────────────────────┴────────────────────┴────────────────────┘

📈 IMPACT IMPUTATION
   Lignes hors barème corrigées     : 115,504
   Lignes hors barème NON corrigées : 332,098
   Total lignes hors barème         : 447,602

📉 

## 5. Export Excel


In [ ]:
# Export des lignes corrigées (agents éligibles, mois hors barème uniquement)

export_df = (
    base_corrigee
    .filter(
        pl.col("eligible_imputation") &
        pl.col("Statut").is_in(statuts_incoherents)
    )
    .select([
        "cle_unique", "matricule", "nom", "GRADE", "emploi", "CODE_ORGANISME", "organisme",
        "mois_annee", "annee", "mois", "Statut",
        "montant_brut_bareme", "montant_corrige",
        "min_bareme", "max_bareme", "nb_incoh"
    ])
)

export_path = "resultats_imputation_forward.xlsx"
export_df.to_pandas().to_excel(export_path, index=False, engine="openpyxl")
print(f"✅ Fichier Excel exporté : {export_path} ({len(export_df):,} lignes)")


✅ Fichier Excel exporté : resultats_imputation_forward.xlsx (115,504 lignes)
